# Vectorize

In [1]:
import lancedb
import pandas as pd

db = lancedb.connect(uri = "vector_database")
db

LanceDBConnection(uri='c:\\Users\\edvin\\.vscode\\ELLA_fab4\\explorations\\vector_database')

In [2]:
db.uri

'c:\\Users\\edvin\\.vscode\\ELLA_fab4\\explorations\\vector_database'

In [3]:
import json
with open("../data/brottsbalken_clean.json", encoding="utf-8") as file:
    data = json.load(file)

df = pd.DataFrame(data)
df

,paragraph,text,law_references,chapter_number,title
0,1 §,Brott är en gärning som är beskriven i denna b...,[_Lag (1994:458)_.],1 kap.,Om brott och brottspåföljder
1,2 §,"En gärning skall, om inte annat är särskilt fö...",[_Lag (1994:458)_.],1 kap.,Om brott och brottspåföljder
2,3 §,Med påföljd för brott förstås i denna balk str...,[_Lag (2026:253)_.],1 kap.,Om brott och brottspåföljder
3,4 §,Om användningen av straffen gäller vad i bestä...,[_Lag (1988:942)_.],1 kap.,Om brott och brottspåföljder
4,5 §,Fängelse är att anse som ett svårare straff än...,[_Lag (2026:253)_.],1 kap.,Om brott och brottspåföljder
...,...,...,...,...,...
551,16 §,En begäran om omprövning skall vara skriftlig ...,[_Lag (2005:967)_.],38 kap.,Rättegångsbestämmelser m.m.
552,17 §,Kriminalvården prövar om skrivelsen med begära...,[_Lag (2005:967)_.],38 kap.,Rättegångsbestämmelser m.m.
553,18 §,Beslut som avses i 14 § överklagas till den fö...,[_Lag (2009:776)_.],38 kap.,Rättegångsbestämmelser m.m.
554,19 §,Mål om uppskjuten villkorlig frigivning enligt...,[_Lag (2021:249)_.],38 kap.,Rättegångsbestämmelser m.m.


In [4]:
def build_embed_text(row):
    parts = [
        f"{row['chapter_number']}  {row['title']}",
        f"{row['paragraph']} ",
        row['text'],
        f"{row['law_references']} "
    ]


    return " | ".join(p for p in parts if p)

df["embed_text"] = df.apply(build_embed_text, axis=1)

df

,paragraph,text,law_references,chapter_number,title,embed_text
0,1 §,Brott är en gärning som är beskriven i denna b...,[_Lag (1994:458)_.],1 kap.,Om brott och brottspåföljder,1 kap. Om brott och brottspåföljder | 1 § | ...
1,2 §,"En gärning skall, om inte annat är särskilt fö...",[_Lag (1994:458)_.],1 kap.,Om brott och brottspåföljder,1 kap. Om brott och brottspåföljder | 2 § | ...
2,3 §,Med påföljd för brott förstås i denna balk str...,[_Lag (2026:253)_.],1 kap.,Om brott och brottspåföljder,1 kap. Om brott och brottspåföljder | 3 § | ...
3,4 §,Om användningen av straffen gäller vad i bestä...,[_Lag (1988:942)_.],1 kap.,Om brott och brottspåföljder,1 kap. Om brott och brottspåföljder | 4 § | ...
4,5 §,Fängelse är att anse som ett svårare straff än...,[_Lag (2026:253)_.],1 kap.,Om brott och brottspåföljder,1 kap. Om brott och brottspåföljder | 5 § | ...
...,...,...,...,...,...,...
551,16 §,En begäran om omprövning skall vara skriftlig ...,[_Lag (2005:967)_.],38 kap.,Rättegångsbestämmelser m.m.,38 kap. Rättegångsbestämmelser m.m. | 16 § |...
552,17 §,Kriminalvården prövar om skrivelsen med begära...,[_Lag (2005:967)_.],38 kap.,Rättegångsbestämmelser m.m.,38 kap. Rättegångsbestämmelser m.m. | 17 § |...
553,18 §,Beslut som avses i 14 § överklagas till den fö...,[_Lag (2009:776)_.],38 kap.,Rättegångsbestämmelser m.m.,38 kap. Rättegångsbestämmelser m.m. | 18 § |...
554,19 §,Mål om uppskjuten villkorlig frigivning enligt...,[_Lag (2021:249)_.],38 kap.,Rättegångsbestämmelser m.m.,38 kap. Rättegångsbestämmelser m.m. | 19 § |...


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 556 entries, 0 to 555
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   paragraph       556 non-null    object
 1   text            556 non-null    object
 2   law_references  556 non-null    object
 3   chapter_number  556 non-null    object
 4   title           556 non-null    object
 5   embed_text      556 non-null    object
dtypes: object(6)
memory usage: 26.2+ KB


In [6]:
df["embed_text"][0]

"1 kap.  Om brott och brottspåföljder | 1 §  | Brott är en gärning som är beskriven i denna balk eller i annan lag eller författning och för vilken straff som sägs nedan är föreskrivet. | ['_Lag (1994:458)_.'] "

In [ ]:
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from utils.constants import EMBEDDING_MODEL

embedding_model = get_registry().get("cohere").create(name=EMBEDDING_MODEL)

class ParagrafModel(LanceModel):
    # LanceDB läser embed_text → skickar till Cohere → får tillbaka en vektor
    # och sparar den automagiskt i vector-kolumnen.
    embed_text: str = embedding_model.SourceField()
    vector: Vector(embedding_model.ndims()) = embedding_model.VectorField()

    # Metadata – sparas som vanliga kolumner så vi kan filtrera på dem senare
    chapter_number: str
    title: str
    law_references: list[str]
    paragraph: str
    text: str  # originaltexten (ej den sammansatta embed_text)



ImportError: cannot import name 'EMBEDDING_MODEL' from 'utils.constants' (c:\Users\edvin\.vscode\ELLA_fab4\explorations\utils\constants.py)

In [ ]:

# lancedb.connect() skapar en lokal mapp "./lancedb" 
# där all data sparas som filer
db = lancedb.connect("./lancedb")

# Skapar tabellen "brottsbalken" med strukturen vi definierade ovan
# mode="overwrite" = om tabellen redan finns, skriv över den (bra under utveckling)
table = db.create_table("brottsbalken", schema=ParagrafModel, mode="overwrite")

# Stoppar in alla rader. LanceDB anropar Cohere automatiskt för varje rad
# och fyller i vector-kolumnen. Det här steget tar lite tid (~553 anrop till Cohere)
table.add(df[[
    "embed_text",
    "chapter_number", "title", "law_references", "paragraph", "text"
]].to_dict(orient="records"))

print(f"{table.count_rows()} paragrafer indexerade")

556 paragrafer indexerade


In [ ]:
table.search("vad är straffet för mord") \
    .limit(5) \
    .to_pandas()[["chapter_number", "title", "paragraph", "text", "law_references", "_distance"]]

,chapter_number,title,paragraph,text,law_references,_distance
0,3 kap.,Om brott mot liv och hälsa,1 §,"Den som berövar annan livet, döms för mord til...",[_Lag (2019:805)_.],0.658175
1,13 kap.,Om allmänfarliga brott,1 §,"Om någon anlägger brand, som innebär fara för ...",[_Lag (1993:207)_.],0.668070
2,13 kap.,Om allmänfarliga brott,2 §,"Är brott som avses i 1 § att anse som grovt, d...",[_Lag (2009:396)_.],0.685560
3,3 kap.,Om brott mot liv och hälsa,2 §,Är brott som i 1 § sägs med hänsyn till de oms...,[],0.688884
4,3 kap.,Om brott mot liv och hälsa,7 §,"Den som av oaktsamhet orsakar annans död, döms...",[_Lag (2010:370)_.],0.747424


In [ ]:
table.search("ofredande") \
    .where("chapter_number = '6'") \
    .limit(5) \
    .to_pandas()[[ "paragraph", "text", "_distance"]]

,paragraph,text,_distance
